# 3.1 · 工具（Tools）：模型可调用的能力

**Tools: capabilities the model can invoke**

> **学完本课你能：** 用精确的模式定义 MCP 工具，返回文本、媒体、链接和结构化内容，报告模型能够从中恢复的错误，并让宿主的工具列表保持最新。

官方阅读材料：[Understanding MCP servers](https://modelcontextprotocol.io/docs/2026-07-28/learn/server-concepts) · [Spec: Tools](https://modelcontextprotocol.io/specification/2026-07-28/server/tools)

从上到下依次运行单元格（`Shift+Enter`），然后改动代码再重新运行。所有内容都在本地运行：服务器位于当前内核、临时目录或 localhost 上。调用真实大语言模型（LLM）的单元格只有在设置了 `ANTHROPIC_API_KEY` 时才会执行，否则自动跳过。

### 1. 一个服务器，每种结果类型一个工具
`MCPServer` 从函数中推导出一切：名称、docstring → `description`、类型注解 → `inputSchema`，以及（对于有类型的返回值）`outputSchema` + `structuredContent`（结构化内容）。`title` 和 `annotations` 是装饰器参数。我们关闭了 SDK 的服务器端日志，让下面的错误演示保持清晰易读。

In [ ]:
import base64
import json
import logging
import struct
import zlib

from pydantic import BaseModel, Field

import mcp_types as types
from mcp import Client
from mcp.server import MCPServer
from mcp.server.mcpserver import Context, Image
from mcp.server.mcpserver.exceptions import ToolError

logging.getLogger("mcp").setLevel(logging.CRITICAL)


def show(model, limit=1500):
    """Print a pydantic protocol object exactly as it goes on the wire (minus serverInfo noise)."""
    data = model.model_dump(by_alias=True, exclude_none=True, mode="json")
    data.get("_meta", {}).pop("io.modelcontextprotocol/serverInfo", None)
    if not data.get("_meta"):
        data.pop("_meta", None)
    print(json.dumps(data, indent=2, ensure_ascii=False)[:limit])


def tiny_png() -> bytes:
    """A valid 1x1 red PNG, built by hand so the notebook needs no image library."""
    chunk = lambda kind, body: (struct.pack(">I", len(body)) + kind + body
                                + struct.pack(">I", zlib.crc32(kind + body) & 0xFFFFFFFF))
    return (b"\x89PNG\r\n\x1a\n" + chunk(b"IHDR", struct.pack(">IIBBBBB", 1, 1, 8, 2, 0, 0, 0))
            + chunk(b"IDAT", zlib.compress(b"\x00\xff\x00\x00")) + chunk(b"IEND", b""))


class Weather(BaseModel):
    temperature: float = Field(description="Temperature in celsius")
    conditions: str
    humidity: float = Field(description="Humidity percentage")


server = MCPServer("travel")


@server.tool()
def greet(name: str) -> str:
    """Say hello. Returns plain text."""
    return f"Hello, {name}!"


@server.tool(title="Weather Data Retriever",
             annotations=types.ToolAnnotations(readOnlyHint=True, openWorldHint=True))
def get_weather_data(location: str) -> Weather:
    """Get current weather data for a location (city name or zip code)."""
    return Weather(temperature=22.5, conditions="Partly cloudy", humidity=65)


@server.tool(annotations=types.ToolAnnotations(readOnlyHint=True))
def render_chart() -> list[types.ContentBlock]:
    """Render the forecast chart as an image meant for the user, plus a caption for the model."""
    return [
        types.ImageContent(type="image", data=base64.b64encode(tiny_png()).decode(), mime_type="image/png",
                           annotations=types.Annotations(audience=["user"], priority=0.9)),
        types.TextContent(type="text", text="Chart rendered: 7-day forecast, mostly sunny."),
    ]


@server.tool()
def find_itinerary(trip: str) -> list[types.ContentBlock]:
    """Return a link to the itinerary file and the packing list inline."""
    return [
        types.ResourceLink(type="resource_link", uri=f"trips://{trip}/itinerary.md", name="itinerary.md",
                           description="Full day-by-day plan", mime_type="text/markdown"),
        types.EmbeddedResource(type="resource", resource=types.TextResourceContents(
            uri=f"trips://{trip}/packing.txt", mime_type="text/plain", text="passport\nadapter\nsunscreen")),
    ]


@server.tool(annotations=types.ToolAnnotations(readOnlyHint=False, destructiveHint=True, idempotentHint=False))
def cancel_booking(booking_id: str) -> str:
    """Cancel a booking. Irreversible."""
    return f"Booking {booking_id} cancelled."


print("tools registered:", [t.name for t in await server.list_tools()])

### 2. 发现：模型将会看到什么
`tools/list` 按注册顺序（确定性的）返回工具定义（Tool Definition）。看看生成的 `inputSchema`、从 pydantic 模型推导出的 `outputSchema`，以及注解。还要留意结果上的 `ttlMs` / `cacheScope`（第 3.4 课）。

In [ ]:
async with Client(server) as client:
    listing = await client.list_tools()
    print("capabilities.tools:", client.server_capabilities.tools)
    print("order:", [t.name for t in listing.tools], "| ttlMs:", listing.ttl_ms, "| cacheScope:", listing.cache_scope)
    show(next(t for t in listing.tools if t.name == "get_weather_data"), limit=2500)

显示名称的优先级是 `title` → `annotations.title` → `name`。宿主（Host）会像这样给按钮加标签：

In [ ]:
def display_name(tool: types.Tool) -> str:
    return tool.title or (tool.annotations.title if tool.annotations else None) or tool.name


async with Client(server) as client:
    for t in (await client.list_tools()).tools:
        a = t.annotations
        hints = a.model_dump(by_alias=True, exclude_none=True) if a else {}
        print(f"{display_name(t):<24} hints={hints}")

### 3. 调用每个工具并读取内容块
纯文本、结构化内容（Structured Content）（及其文本镜像）、标注为给用户看的图片、资源链接和嵌入式资源。

In [ ]:
async with Client(server) as client:
    for name, args in [("greet", {"name": "Ada"}), ("get_weather_data", {"location": "Barcelona"}),
                       ("render_chart", {}), ("find_itinerary", {"trip": "bcn-2026"})]:
        result = await client.call_tool(name, args)
        print(f"--- tools/call {name}")
        show(result, limit=900)

对于 `get_weather_data`，SDK 同时填充了 `structuredContent`（给程序用）<strong>和</strong>一个包含相同 JSON 的文本块（给模型和旧客户端用）——这正是规范中的 SHOULD。对于 `greet`，`str` 返回值被包装成 `{"result": ...}`，因为在这个 SDK 中，函数的返回类型总是产生一个对象形态的模式（Schema）。

### 4. 破坏它：一个谎报输出的服务器
客户端（Client）应当（SHOULD）根据 `outputSchema` 校验 `structuredContent`。SDK 的客户端在 `call_tool` 内部完成这一步。这里我们把一个与模式不符的伪造结果交给它的校验器，看看有 bug 的或恶意的服务器会触发什么。

In [ ]:
async with Client(server) as client:
    await client.list_tools()  # the client remembers each tool's outputSchema
    forged = types.CallToolResult(content=[types.TextContent(type="text", text="it's warm")],
                                  structured_content={"temperature": "warm", "conditions": "sunny", "humidity": 65})
    try:
        await client.session.validate_tool_result("get_weather_data", forged)
    except RuntimeError as exc:
        print("client rejected the result:\n", str(exc).splitlines()[0])

### 5. 两类错误
四种失败模式。观察哪些变成了 `isError` 结果（工具执行错误（Tool Execution Error），模型可见），以及每条消息向模型透露了多少信息。

In [ ]:
@server.tool()
def book_flight(origin: str, destination: str, date: str) -> str:
    """Book a flight. date must be YYYY-MM-DD and in the future."""
    if not date[:4].isdigit() or date < "2026-07-28":
        raise ToolError(f"Invalid departure date {date!r}: must be YYYY-MM-DD and in the future.")
    return f"Booked {origin}->{destination} on {date}"


@server.tool()
def flaky_api() -> str:
    """Calls an upstream API that crashes."""
    raise ConnectionError("upstream 10.0.0.7:5432 refused connection (password=hunter2)")


async with Client(server) as client:
    cases = [("business error (ToolError)", "book_flight", {"origin": "NYC", "destination": "BCN", "date": "08/08/2025"}),
             ("unexpected exception", "flaky_api", {}),
             ("arguments fail inputSchema", "book_flight", {"origin": "NYC"}),
             ("unknown tool", "teleport", {})]
    for label, name, args in cases:
        r = await client.call_tool(name, args)
        print(f"{label:<28} isError={r.is_error} -> {r.content[0].text.splitlines()[0][:110]}")

注意 SDK 的两个行为。`ToolError` 的消息会到达模型（可据以行动），而意外的异常则变成一个笼统的 “Error executing tool flaky_api”——主机地址和密码之类的内部细节<strong>不会</strong>泄露，这正是“清理工具输出”的要求。另外，这个 SDK 也把未知工具报告为 `isError`，而规范把它列为协议错误。

### 6. 线路上的协议错误
为了看到真正的 JSON-RPC 错误（协议错误（Protocol Error）），我们要绕到 SDK 下面：把服务器作为 stdio 子进程运行，并发送一个违反 `CallToolRequest` 模式的 `tools/call`（没有 `name`）。

In [ ]:
import subprocess
import sys
import tempfile
import textwrap
from pathlib import Path

workdir = Path(tempfile.mkdtemp())
(workdir / "weather_server.py").write_text(textwrap.dedent('''
    from mcp.server import MCPServer
    server = MCPServer("weather")

    @server.tool()
    def get_weather(location: str) -> str:
        """Get current weather information for a location."""
        return f"Current weather in {location}: 22C, partly cloudy"

    if __name__ == "__main__":
        server.run(transport="stdio")
'''))
proc = subprocess.Popen([sys.executable, str(workdir / "weather_server.py")], stdin=subprocess.PIPE,
                        stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, text=True)
META = {"io.modelcontextprotocol/protocolVersion": "2026-07-28",
        "io.modelcontextprotocol/clientCapabilities": {},
        "io.modelcontextprotocol/clientInfo": {"name": "by-hand", "version": "0.1"}}


def rpc(id_, method, params):
    proc.stdin.write(json.dumps({"jsonrpc": "2.0", "id": id_, "method": method, "params": params}) + "\n")
    proc.stdin.flush()
    reply = json.loads(proc.stdout.readline())
    reply.get("result", {}).pop("_meta", None)   # hide serverInfo noise
    print(f">>> {method} {json.dumps({k: v for k, v in params.items() if k != '_meta'})}")
    print("<<<", json.dumps(reply), "\n")


rpc(1, "tools/call", {"name": "get_weather", "arguments": {"location": "New York"}, "_meta": META})
rpc(2, "tools/call", {"arguments": {"location": "New York"}, "_meta": META})   # malformed: no name
proc.stdin.close()
_ = proc.wait(timeout=10)

### 7. 工具名称
SDK 内置了规范中的命名检查（它只在注册时<em>发出警告</em>）。试几个名称。

In [ ]:
from mcp.shared.tool_name_validation import validate_tool_name

for name in ["getUser", "DATA_EXPORT_v2", "admin.tools.list", "search flights", "créer_événement", "x" * 129]:
    r = validate_tool_name(name)
    print(f"{name[:30]!r:<34} valid={r.is_valid!s:<5} {r.warnings[:1]}")

### 8. 使用显式句柄的有状态工具
没有会话（Session），所以状态保存在一个不透明句柄（State Handle）之下，由模型在调用之间携带。未知的句柄是一个 `isError` 结果，它告诉模型如何恢复。

In [ ]:
import uuid

baskets: dict[str, list[str]] = {}
shop = MCPServer("shop")


class BasketRef(BaseModel):
    basket_id: str


@shop.tool()
def create_basket() -> BasketRef:
    """Create an empty basket and return its id. Baskets expire after 24 hours of inactivity."""
    basket_id = f"bsk_{uuid.uuid4().hex[:12]}"
    baskets[basket_id] = []
    return BasketRef(basket_id=basket_id)


@shop.tool()
def add_item(basket_id: str, sku: str) -> list[str]:
    """Add a product to an existing basket; returns the basket contents."""
    if basket_id not in baskets:
        raise ToolError(f"Unknown or expired basket {basket_id!r}. Call create_basket to start a new one.")
    baskets[basket_id].append(sku)
    return baskets[basket_id]


async with Client(shop) as client:
    handle = (await client.call_tool("create_basket", {})).structured_content["basket_id"]
    print("handle:", handle)
    print((await client.call_tool("add_item", {"basket_id": handle, "sku": "TENT-2P"})).structured_content)
    bad = await client.call_tool("add_item", {"basket_id": "bsk_000000000000", "sku": "STOVE"})
    print("isError:", bad.is_error, "->", bad.content[0].text)

### 9. 动态工具与 `notifications/tools/list_changed`
一个工具可以改变工具集合。在 2026-07-28 中，客户端只有在以 `toolsListChanged` 打开了 `subscriptions/listen` 流时才会得知这一变化（第 3.4 课会深入讲解）。调用 `add_tool` 时 `MCPServer` 不会自动发出通知（Notification）；由工具通过 `ctx.notify_tools_changed()` 发布该事件。

In [ ]:
@server.tool()
async def enable_hotels(ctx: Context) -> str:
    """Unlock hotel-booking tools for this deployment."""
    def search_hotels(city: str) -> list[str]:
        """Search hotels in a city."""
        return [f"Hotel Arts {city}", f"Casa {city}"]
    server.add_tool(search_hotels)
    await ctx.notify_tools_changed()
    return "hotel tools enabled"


async with Client(server) as client:
    before = [t.name for t in (await client.list_tools()).tools]
    async with client.listen(tools_list_changed=True) as sub:
        print("listening, honored filter:", sub.honored.model_dump(exclude_none=True, by_alias=True))
        await client.call_tool("enable_hotels", {})
        event = await anext(sub)
        print("event:", event)
        after = [t.name for t in (await client.list_tools()).tools]
    print("new tools:", sorted(set(after) - set(before)))

### 10. 宿主端的审批关卡
协议把用户体验留给宿主（Host），但规范说人应当（SHOULD）能够拒绝调用，并且除非服务器受信任，否则注解是不可信的。下面是一个最小策略：<strong>仅对受信任的服务器</strong>自动批准只读工具，其他情况一律询问（由一个脚本化的“用户”模拟）。

In [ ]:
TRUSTED_SERVERS = {"travel"}


def needs_approval(server_name: str, tool: types.Tool) -> bool:
    hints = tool.annotations or types.ToolAnnotations()
    if server_name not in TRUSTED_SERVERS:
        return True                      # never trust hints from unknown servers
    return not hints.read_only_hint      # trusted + read-only -> no dialog


def scripted_user(tool_name: str, args: dict) -> bool:
    print(f"   [dialog] Allow {tool_name}({args})?", "-> DENY" if tool_name.startswith("cancel") else "-> allow")
    return not tool_name.startswith("cancel")


async def host_call(client, server_name, name, args):
    tool = next(t for t in (await client.list_tools()).tools if t.name == name)
    if needs_approval(server_name, tool) and not scripted_user(name, args):
        return f"{name}: denied by user (the model is told the call was refused)"
    text = (await client.call_tool(name, args)).content[0].text
    return f"{name}: " + " ".join(text.split())[:70]


async with Client(server) as client:
    for server_name, name, args in [("travel", "get_weather_data", {"location": "Rome"}),
                                    ("travel", "cancel_booking", {"booking_id": "B-42"}),
                                    ("random-server", "get_weather_data", {"location": "Rome"})]:
        print(f"{server_name}/{name}")
        print("  ", await host_call(client, server_name, name, args))

### 动手练习
1. 添加一个工具（Tool）`convert_currency(amount: float, frm: str, to: str)`，其返回类型是一个带有 `amount` 和 `rate` 的 pydantic 模型。查看它的 `outputSchema`，然后让它对不支持的货币抛出 `ToolError`，并检查该消息是否以 `isError: true` 到达客户端。
2. 给 `get_weather_data` 在 `annotations` 中加一个显式的 `title`，<em>同时</em>保留装饰器的 `title`。`display_name` 会选哪一个？去掉装饰器的 title 后再检查一次。
3. 在第 6 节中，通过原始 stdio 为一个不存在的工具名称发送 `tools/call`。把得到的答复与规范中的 `-32602 Unknown tool` 示例进行比较，并决定你的宿主应如何处理这两种形态。
4. 扩展第 8 节，添加一个删除购物篮的 `checkout(basket_id)` 工具，然后演示用同一个句柄再次调用 `add_item` 会产生一个可恢复的 `isError` 结果。